# Лабораторная работа 1 (Вариат 13)
**Тема.** РЕШЕНИЕ СИСТЕМ ЛИНЕЙНЫХ АЛГЕБРАИЧЕСКИХ УРАВНЕНИЙ (СЛАУ)
<br>
Составить программу решения СЛАУ порядка $ n $ и решить систему линейных уравнений пятого порядка с трехдиагональной симметричной матрицей вида:
$$
\begin{vmatrix}
q & 1 & 0 & 0 & 0\\
1 & -2 & 1 & 0 & 0\\
0 & 1 & -2 & 1 & 0\\
0 & 0 & 1 & -2 & 1\\
0 & 0 & 0 & 1 & q
\end{vmatrix} \times
\begin{vmatrix}
x_1\\
x_2\\
x_3\\
x_4\\
x_5
\end{vmatrix} = 
\begin{vmatrix}
0\\
d\\
d\\
d\\
0
\end{vmatrix}
$$

В вариантах, использующих прямые методы вычислить невязку.

Вариант предполагает решение методом Гаусса. Возьмем значения $ d $ и $ q $.

In [1]:
import numpy as np

q = -2.86
d = -7

Теперь объявим матрицы

In [2]:
A = np.array([
    [q,  1,  0,  0,  0],
    [1, -2,  1,  0,  0],
    [0,  1, -2,  1,  0],
    [0,  0,  1, -2,  1],
    [0,  0,  0,  1,  q]
], dtype=float)

b = np.array([0, d, d, d, 0], dtype=float)

print(A)
print(b)

[[-2.86  1.    0.    0.    0.  ]
 [ 1.   -2.    1.    0.    0.  ]
 [ 0.    1.   -2.    1.    0.  ]
 [ 0.    0.    1.   -2.    1.  ]
 [ 0.    0.    0.    1.   -2.86]]
[ 0. -7. -7. -7.  0.]


Теперь реализуем метод Гаусса для решения данной системы. Сформируем расширенную матрицу

In [3]:
n = len(b)
matrix = np.hstack((A.astype(float), b.reshape(-1, 1).astype(float)))
print(matrix)

[[-2.86  1.    0.    0.    0.    0.  ]
 [ 1.   -2.    1.    0.    0.   -7.  ]
 [ 0.    1.   -2.    1.    0.   -7.  ]
 [ 0.    0.    1.   -2.    1.   -7.  ]
 [ 0.    0.    0.    1.   -2.86  0.  ]]


**Прямой ход.** Приведение к ступенчатому виду

In [4]:
for i in range(n):
    max_row = np.argmax(np.abs(matrix[i:, i])) + i
    matrix[i], matrix[max_row] = matrix[max_row].copy(), matrix[i].copy()

    if abs(matrix[i, i]) < 1e-10:
        raise ValueError("Матрица вырождена или имеет бесконечное множество решений.")
    for j in range(i + 1, n):
        ratio = matrix[j, i] / matrix[i, i]
        matrix[j, i:] -= ratio * matrix[i, i:]

**Обратный ход.** Нахождение переменных

In [5]:
x = np.zeros(n)
for i in range(n - 1, -1, -1):
    x[i] = (matrix[i, -1] - np.dot(matrix[i, i+1:n], x[i+1:n])) / matrix[i, i]

Итог

In [6]:
x

array([ 5.64516129, 16.14516129, 19.64516129, 16.14516129,  5.64516129])

Вычислим невязку для оценки точности решения СЛАУ. Невязка — это вектор, который показывает, насколько точно полученное решение $x$ удовлетворяет исходному уравнению $A \cdot x=b$. Вектор невязки $r$ вычисляется по формуле:
$$r=b−Ax$$
где:
   - $A$ — матрица коэффициентов;
   - $x$ — найденное решение;
   - $b$ — вектор свободных членов.

In [7]:
residual = b - np.dot(A, x)
print(f"Вектор невязки:\n{residual}")

Вектор невязки:
[ 0.00000000e+00  0.00000000e+00  0.00000000e+00 -8.88178420e-16
  7.72428716e-16]


In [8]:
residual_norm = np.linalg.norm(residual)
print(f"\nНорма невязки: {residual_norm:.2e}")


Норма невязки: 1.18e-15


Как видно, норма невязки имеет порядок $10^{-15}$, что является теоретическим максимумом точности для типа `float64`